<a href="https://colab.research.google.com/github/ounissinour/PolypVision/blob/main/milestone1/notebooks/M1_Reproduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Milestone 1 – Reproduce PraNet and Polyp-PVT

Runs the official test scripts of both models on the five test sets and computes mean Dice /
mean IoU with `pysodmetrics`, the same definition as the papers' MATLAB toolbox.

Runtime: GPU (a T4 is enough). About 15 minutes end to end, most of it downloads.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())

## Settings

`USE_DRIVE = True` keeps the downloaded zips, weights and the results in your Google Drive, so a
second run skips the downloads. Data and prediction maps always stay on the Colab disk: unzipping
thousands of small files onto the Drive mount is slow and can stop half-way.

In [ ]:
import os

REPO_URL = "https://github.com/ounissinour/PolypVision"
REPO_BRANCH = "main"
PRANET_COMMIT = "95f12bd4e32ec954b5069654abae7de9473fd1f5"   # official PraNet repo
PVT_COMMIT = "cd71fbf9fcd8c32cc0f33699f4e0186073c64dc4"      # official Polyp-PVT repo
USE_DRIVE = True

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/polyp-m1"
else:
    WORK = "/content/work"
DATA = "/content/data"                     # unzipped datasets
MAPS = "/content/maps"                     # prediction maps of both models
M1 = "/content/PolypVision/milestone1"     # this folder of our repository

os.makedirs(WORK, exist_ok=True)
os.environ.update(WORK=WORK, DATA=DATA, MAPS=MAPS, M1=M1)   # so the !shell cells can use them
print("work folder:", WORK)

In [ ]:
!pip install -q gdown pysodmetrics timm tabulate

## Clone our repository and the two official repositories (at fixed commits)

In [ ]:
%cd /content
!rm -rf PolypVision PraNet Polyp-PVT
!git clone -q --branch $REPO_BRANCH $REPO_URL PolypVision
!git clone -q https://github.com/DengPingFan/PraNet && git -C PraNet checkout -q $PRANET_COMMIT
!git clone -q https://github.com/DengPingFan/Polyp-PVT && git -C Polyp-PVT checkout -q $PVT_COMMIT
!git -C PraNet log -1 --format='PraNet    %h %cd' && git -C Polyp-PVT log -1 --format='Polyp-PVT %h %cd'

## Download data and trained models

Google Drive links from the two official READMEs, fetched with `gdown` by `scripts/download.sh`.
Expected test-set sizes: CVC-300 60, CVC-ClinicDB 62, Kvasir 100, CVC-ColonDB 380, ETIS 196.

In [ ]:
!bash $M1/scripts/download.sh $WORK $DATA

## Patch the official test scripts

PraNet's `MyTest.py` uses `scipy.misc.imsave`, which no longer exists; it is replaced by
`cv2.imwrite(..., res*255)` exactly as Polyp-PVT's `Test.py` does. `scripts/patch_repos.py`
lists the few other compatibility changes; none of them changes the predictions.

In [ ]:
!python $M1/scripts/patch_repos.py /content/PraNet /content/Polyp-PVT
!git -C /content/PraNet diff --stat && git -C /content/Polyp-PVT diff --stat

## Put data and weights where each test script expects them

In [ ]:
%%bash
set -e
mkdir -p /content/PraNet/data /content/PraNet/snapshots/PraNet_Res2Net
ln -sfn $DATA/TestDataset /content/PraNet/data/TestDataset
ln -sf  $WORK/weights/PraNet-19.pth /content/PraNet/snapshots/PraNet_Res2Net/PraNet-19.pth

mkdir -p /content/Polyp-PVT/dataset /content/Polyp-PVT/model_pth /content/Polyp-PVT/pretrained_pth
ln -sfn $DATA/TestDataset /content/Polyp-PVT/dataset/TestDataset
ln -sf  $WORK/weights/PolypPVT.pth  /content/Polyp-PVT/model_pth/PolypPVT.pth
ln -sf  $WORK/weights/pvt_v2_b2.pth /content/Polyp-PVT/pretrained_pth/pvt_v2_b2.pth
ls -l /content/PraNet/snapshots/PraNet_Res2Net /content/Polyp-PVT/model_pth /content/Polyp-PVT/pretrained_pth

## Run the official test scripts (default test size 352)

In [ ]:
!cd /content/PraNet && python -W ignore MyTest.py
!cd /content/Polyp-PVT && python -W ignore Test.py

Collect the prediction maps in one place and count them.

In [ ]:
%%bash
mkdir -p $MAPS
rm -rf $MAPS/PraNet $MAPS/Polyp-PVT
cp -r /content/PraNet/results/PraNet $MAPS/PraNet
cp -r /content/Polyp-PVT/result_map/PolypPVT $MAPS/Polyp-PVT
for model in PraNet Polyp-PVT; do
  for dataset in CVC-300 CVC-ClinicDB Kvasir CVC-ColonDB ETIS-LaribPolypDB; do
    printf '%-10s %-18s %4s maps\n' $model $dataset $(ls $MAPS/$model/$dataset | wc -l)
  done
done

## Evaluate

`meanDice` / `meanIoU` are the paper metrics (mean over 256 thresholds). `Dice@0.5` is a single
0.5 threshold, printed only for reference: it is always a bit higher.

In [ ]:
%cd $M1
!mkdir -p results
!python src/evaluate.py --model PraNet    --pred_root $MAPS/PraNet    --gt_root $DATA/TestDataset --out results
!python src/evaluate.py --model Polyp-PVT --pred_root $MAPS/Polyp-PVT --gt_root $DATA/TestDataset --out results

## Evaluator check

Score the authors' own published prediction maps (links in their READMEs) with the same code.
This should give the paper numbers exactly; `results/authors_maps_check/` holds the expected output.

In [ ]:
%%bash
cd $WORK/downloads
[ -f pranet_maps.zip ] || gdown -q 1CJ6CTUdenumgiKXieuKXFohefRJwyFPY -O pranet_maps.zip
[ -f pvt_maps.zip ]    || gdown -q 1L0pFFmd9fbqnJnBwrM9cEV5nlRS32mbQ -O pvt_maps.zip
rm -rf /content/authors_maps && mkdir -p /content/authors_maps
unzip -q -o pranet_maps.zip -d /content/authors_maps && unzip -q -o pvt_maps.zip -d /content/authors_maps

cd $M1 && mkdir -p results/authors_maps_check
python src/evaluate.py --model PraNet    --pred_root /content/authors_maps/PraNet              --gt_root $DATA/TestDataset --out results/authors_maps_check
python src/evaluate.py --model Polyp-PVT --pred_root /content/authors_maps/result_map/PolypPVT --gt_root $DATA/TestDataset --out results/authors_maps_check

## Compare with the paper tables

In [ ]:
!python src/compare.py --results results

Run the tests (synthetic masks, and the comparison table against the paper values).

In [ ]:
!python -m pytest -q tests

## Figures

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

table = pd.read_csv("results/comparison.csv")
datasets = ["Kvasir", "CVC-ClinicDB", "CVC-ColonDB", "ETIS-LaribPolypDB", "CVC-300"]
models = [("PraNet", "tab:blue"), ("Polyp-PVT", "tab:orange")]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, metric in zip(axes, ["Dice", "IoU"]):
    x = np.arange(len(datasets))
    for k, (model, color) in enumerate(models):
        rows = table[table.model == model].set_index("dataset").loc[datasets]
        offset = (k - 0.5) * 0.4
        ax.bar(x + offset, rows[f"mean{metric}"], 0.38, color=color, label=f"{model} (ours)")
        ax.scatter(x + offset, rows[f"paper_mean{metric}"], color="k", marker="_", s=300, zorder=3,
                   label="paper" if k == 0 else None)
    ax.set_xticks(x)
    ax.set_xticklabels(datasets, rotation=15)
    ax.set_ylim(0.5, 1.0)
    ax.set_title(f"mean {metric}")
    ax.legend(loc="lower left")
plt.tight_layout()
plt.savefig("figures/reproduced_vs_paper.png", dpi=200)
plt.show()

Qualitative check: on each dataset, the image where the two models disagree most.

In [ ]:
import glob
import cv2

pranet = pd.read_csv("results/PraNet_per_image.csv")
pvt = pd.read_csv("results/Polyp-PVT_per_image.csv")
both = pranet.merge(pvt, on=["dataset", "image"], suffixes=("_pranet", "_pvt"))
both["gap"] = (both["dice_mean_thr_pvt"] - both["dice_mean_thr_pranet"]).abs()
worst = both.loc[both.groupby("dataset")["gap"].idxmax()]

fig, axes = plt.subplots(len(worst), 4, figsize=(12, 3 * len(worst)))
for ax_row, (_, row) in zip(axes, worst.iterrows()):
    stem = os.path.splitext(row.image)[0]
    image = cv2.imread(glob.glob(f"{DATA}/TestDataset/{row.dataset}/images/{stem}.*")[0])[:, :, ::-1]
    mask = cv2.imread(glob.glob(f"{DATA}/TestDataset/{row.dataset}/masks/{stem}.*")[0], 0)
    panels = [
        (image, f"{row.dataset}/{row.image}"),
        (mask, "ground truth"),
        (cv2.imread(f"{MAPS}/PraNet/{row.dataset}/{row.image}", 0), f"PraNet Dice {row.dice_mean_thr_pranet:.2f}"),
        (cv2.imread(f"{MAPS}/Polyp-PVT/{row.dataset}/{row.image}", 0), f"Polyp-PVT Dice {row.dice_mean_thr_pvt:.2f}"),
    ]
    for ax, (img, title) in zip(ax_row, panels):
        ax.imshow(img, cmap=None if img.ndim == 3 else "gray")
        ax.set_title(title, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.savefig("figures/largest_disagreements.png", dpi=150)
plt.show()

## Save results, figures and our prediction maps to the work folder (Drive) so they survive the session

In [ ]:
!mkdir -p $WORK/m1_out && cp -r results figures $MAPS $WORK/m1_out/ && ls $WORK/m1_out $WORK/m1_out/results